# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to load, explore, and analyze the FAIR^2 dataset using the `mlcroissant` library.

### Dataset Source
The dataset schema is described in Croissant format and is accessible at the URL provided below.

In [ ]:
# Ensure `mlcroissant` is installed in this environment
!pip install mlcroissant

## 1. Data Loading
Load the Croissant dataset and print the dataset's name and description. All subsequent steps reference entities only by their `@id` values, per Croissant best practices.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import warnings

# Croissant metadata schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the Croissant dataset
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Explore the available record sets, fields, and their `@id`s. Each record set corresponds to a logical table or resource, and each field represents a data attribute. All are referenced by `@id` for reproducibility.

In [ ]:
# List all record sets in the dataset with their @id and name
print("Available Record Sets:")
for record_set in metadata.record_sets:
    print(f"  RecordSet @id: {record_set['@id']}  name: {record_set.get('name', '(none)')}")
    # List fields within each record set
    if 'field' in record_set:
        print("    Fields:")
        for field in record_set['field']:
            field_obj = metadata.get_entity(field)  # resolves field given @id
            print(f"      Field @id: {field_obj['@id']}  name: {field_obj.get('name', '(none)')}  dataType: {field_obj.get('dataType', '(none)')}")

## 3. Data Extraction
Record sets are referenced by `@id`. This step loads each available record set as a `pandas.DataFrame`, shows their columns, and displays the first few records. Replace the example `record_set_id` in the cell below with one from the output above.

In [ ]:
# List all record_set @id values
record_sets = [rs['@id'] for rs in metadata.record_sets]
dataframes = {}
for record_set_id in record_sets:
    with warnings.catch_warnings():
        warnings.simplefilter('ignore')
        records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"Loaded {len(df)} records for record set @id: {record_set_id}")
    print(f"  Columns: {df.columns.tolist()}")
    print(df.head(), "\n")
# Example: Select first available record set for downstream steps
main_record_set_id = record_sets[0] if record_sets else None
if main_record_set_id:
    print(f"Using record set @id: {main_record_set_id} for analysis.")

## 4. Exploratory Data Analysis (EDA)
As an example, filter records where a numeric field (identified by its `@id`) exceeds a threshold, normalize the field, and group by another field. You may adjust the `numeric_field_id` and `group_field_id` variables using `@id`s from the previous overview.

In [ ]:
# Identify a numeric field for filtering and grouping by a categorical field
# To determine candidates, print columns again if needed
if main_record_set_id:
    df = dataframes[main_record_set_id]
    print(f"Available columns for record set {main_record_set_id}: {df.columns.tolist()}")

    # For illustration, choose the first float/int column for demonstrating EDA
    numeric_field_id = None
    for col in df.columns:
        if pd.api.types.is_numeric_dtype(df[col]):
            numeric_field_id = col
            break
    if not numeric_field_id:
        # fallback: pick column with likely numeric values
        for col in df.columns:
            try:
                _ = pd.to_numeric(df[col])
                numeric_field_id = col
                df[col] = pd.to_numeric(df[col], errors='coerce')
                break
            except:
                continue
    print(f"Using numeric field @id: {numeric_field_id}")

    # Select a group field by inspecting columns - prefer categorical or string fields
    group_field_id = None
    for col in df.columns:
        if col != numeric_field_id:
            if pd.api.types.is_object_dtype(df[col]) or pd.api.types.is_categorical_dtype(df[col]):
                group_field_id = col
                break

    threshold = 10  # Example threshold; adjust per field meaning
    filtered_df = df[df[numeric_field_id] > threshold].copy()
    print(f"Filtered records in {main_record_set_id} with {numeric_field_id} > {threshold}:")
    print(filtered_df[[numeric_field_id]].head())

    # Normalize the numeric field
    filtered_df[f"{numeric_field_id}_normalized"] = (
        filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
    ) / filtered_df[numeric_field_id].std()
    print(f"Normalized {numeric_field_id} for filtered records:")
    print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    if group_field_id:
        grouped_df = filtered_df.groupby(group_field_id, dropna=True)[numeric_field_id].mean()
        print(f"Grouped mean {numeric_field_id} by {group_field_id}:")
        print(grouped_df.head())
else:
    print("No record set loaded for EDA.")

## 5. Visualization
Visualize the distribution of the chosen numeric field and the effect of any grouping. You may adapt or extend visualizations as needed for your analysis.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if main_record_set_id and numeric_field_id:
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field_id].dropna(), kde=True, bins=30)
    plt.title(f"Distribution of {numeric_field_id} in {main_record_set_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Count")
    plt.grid(True)
    plt.show()
    if group_field_id and group_field_id in df.columns:
        plt.figure(figsize=(10,5))
        sns.boxplot(data=df, x=group_field_id, y=numeric_field_id)
        plt.xticks(rotation=45)
        plt.title(f"{numeric_field_id} grouped by {group_field_id}")
        plt.grid(True)
        plt.show()
else:
    print("Not enough information for visualization.")

## 6. Conclusion
- This notebook loaded the FAIR^2 dataset via its Croissant schema and explored available record sets, fields, and sample statistics.
- We demonstrated basic filtering, normalization, grouping, and visual data inspection, all referencing entities by their `@id`.
- You may extend this notebook for deeper domain analysis, modeling, or comparison using the established workflow.